In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("Smart Metro Operation Analytics System Dataset.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.isnull().sum()

# EDA

In [ ]:
# Convert Date to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Clean station names (remove extra spaces and standardize case)
df['From_Station'] = df['From_Station'].str.strip().str.title()
df['To_Station'] = df['To_Station'].str.strip().str.title()

# Handle missing values
df['Ticket_Type'] = df['Ticket_Type'].fillna('Unknown')
df['Remarks'] = df['Remarks'].fillna('None')

In [ ]:
# Create additional features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.day_name()
df['MonthYear'] = df['Date'].dt.to_period('M')

In [ ]:
# Calculate total revenue
df['Total_Revenue'] = df['Fare'] * df['Passengers']

In [ ]:
print("Data after cleaning:")
print(f"Number of unique stations: {df['From_Station'].nunique()}")

In [ ]:
# 1. Basic Statistics Overview
print("=== BASIC STATISTICS ===")
print(df[['Distance_km', 'Fare', 'Cost_per_passenger', 'Passengers', 'Total_Revenue']].describe())

# Create subplots for distribution analysis
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Distance Distribution', 'Fare Distribution', 'Cost per Passenger', 
                   'Passengers Distribution', 'Total Revenue Distribution', 'Monthly Trends'),
    specs=[[{"secondary_y": False}, {"secondary_y": False}, {"secondary_y": False}],
          [{"secondary_y": False}, {"secondary_y": False}, {"secondary_y": False}]]
)

# Distance distribution
fig.add_trace(go.Histogram(x=df['Distance_km'], name='Distance', nbinsx=30), row=1, col=1)

# Fare distribution
fig.add_trace(go.Histogram(x=df['Fare'], name='Fare', nbinsx=30), row=1, col=2)

# Cost per passenger distribution
fig.add_trace(go.Histogram(x=df['Cost_per_passenger'], name='Cost per Passenger', nbinsx=30), row=1, col=3)

# Passengers distribution
fig.add_trace(go.Histogram(x=df['Passengers'], name='Passengers', nbinsx=30), row=2, col=1)

# Total revenue distribution
fig.add_trace(go.Histogram(x=df['Total_Revenue'], name='Total Revenue', nbinsx=30), row=2, col=2)

# Monthly passenger trends
monthly_passengers = df.groupby('Month')['Passengers'].sum().reset_index()
fig.add_trace(go.Scatter(x=monthly_passengers['Month'], y=monthly_passengers['Passengers'], 
                       mode='lines+markers', name='Monthly Passengers'), row=2, col=3)

fig.update_layout(height=600, showlegend=False, title_text="Distribution Analysis")
fig.show()

In [ ]:
# 2. Time Series Analysis
# Monthly trends for key metrics
monthly_data = df.groupby('MonthYear').agg({
    'Passengers': 'sum',
    'Total_Revenue': 'sum',
    'TripID': 'count',
    'Distance_km': 'mean',
    'Fare': 'mean'
}).reset_index()
monthly_data['MonthYear'] = monthly_data['MonthYear'].astype(str)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Monthly Passengers', 'Monthly Revenue', 
                   'Monthly Trips', 'Average Fare Trends'),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
          [{"secondary_y": False}, {"secondary_y": False}]]
)

# Monthly passengers
fig.add_trace(go.Scatter(x=monthly_data['MonthYear'], y=monthly_data['Passengers'], 
                       mode='lines+markers', name='Passengers'), row=1, col=1)

# Monthly revenue
fig.add_trace(go.Scatter(x=monthly_data['MonthYear'], y=monthly_data['Total_Revenue'], 
                       mode='lines+markers', name='Revenue', line=dict(color='green')), row=1, col=2)

# Monthly trips
fig.add_trace(go.Scatter(x=monthly_data['MonthYear'], y=monthly_data['TripID'], 
                       mode='lines+markers', name='Trips', line=dict(color='red')), row=2, col=1)

# Average fare trends
fig.add_trace(go.Scatter(x=monthly_data['MonthYear'], y=monthly_data['Fare'], 
                       mode='lines+markers', name='Avg Fare', line=dict(color='purple')), row=2, col=2)

fig.update_layout(height=600, showlegend=False, title_text="Time Series Analysis - Monthly Trends")
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# 3. STATION NETWORK ANALYSIS
print("\n\n3. STATION NETWORK ANALYSIS")
print("-" * 40)

# Create station network metrics
all_stations = pd.concat([df['From_Station'], df['To_Station']]).unique()
print(f"Total unique stations in network: {len(all_stations)}")

# Station centrality analysis
station_centrality = pd.DataFrame(index=all_stations)
station_centrality['Departures'] = df.groupby('From_Station')['TripID'].count()
station_centrality['Arrivals'] = df.groupby('To_Station')['TripID'].count()
station_centrality['Total_Traffic'] = station_centrality['Departures'] + station_centrality['Arrivals']
station_centrality['Net_Flow'] = station_centrality['Departures'] - station_centrality['Arrivals']

# Passenger volume by station
passenger_flow = pd.DataFrame()
passenger_flow['Departing_Passengers'] = df.groupby('From_Station')['Passengers'].sum()
passenger_flow['Arriving_Passengers'] = df.groupby('To_Station')['Passengers'].sum()
passenger_flow['Total_Passengers'] = passenger_flow['Departing_Passengers'] + passenger_flow['Arriving_Passengers']
passenger_flow['Passenger_Net_Flow'] = passenger_flow['Departing_Passengers'] - passenger_flow['Arriving_Passengers']

station_analysis = station_centrality.merge(passenger_flow, left_index=True, right_index=True, how='outer')
station_analysis = station_analysis.fillna(0).sort_values('Total_Passengers', ascending=False)

print("\nTop 15 Stations by Passenger Volume:")
print(station_analysis.head(15))

# Station classification
def classify_station(row):
    if row['Total_Passengers'] > station_analysis['Total_Passengers'].quantile(0.8):
        return 'Hub'
    elif row['Total_Passengers'] > station_analysis['Total_Passengers'].quantile(0.5):
        return 'Major'
    elif row['Total_Passengers'] > station_analysis['Total_Passengers'].quantile(0.2):
        return 'Medium'
    else:
        return 'Minor'

station_analysis['Station_Type'] = station_analysis.apply(classify_station, axis=1)

# Visualize station network
fig_stations = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Station Traffic Distribution', 'Station Types Distribution',
                   'Top 20 Stations by Passenger Volume', 'Passenger Net Flow Analysis'),
    specs=[[{"type": "histogram"}, {"type": "pie"}],
           [{"type": "bar"}, {"type": "bar"}]]
)

# Station traffic distribution
fig_stations.add_trace(
    go.Histogram(x=station_analysis['Total_Passengers'], name='Passenger Distribution'),
    row=1, col=1
)

# Station types
station_type_counts = station_analysis['Station_Type'].value_counts()
fig_stations.add_trace(
    go.Pie(labels=station_type_counts.index, values=station_type_counts.values, name='Station Types'),
    row=1, col=2
)

# Top 20 stations
top_20_stations = station_analysis.head(20)
fig_stations.add_trace(
    go.Bar(x=top_20_stations.index, y=top_20_stations['Total_Passengers'], 
           name='Top 20 Stations'),
    row=2, col=1
)

# Net flow analysis
net_flow_stations = station_analysis.nlargest(10, 'Passenger_Net_Flow')
fig_stations.add_trace(
    go.Bar(x=net_flow_stations.index, y=net_flow_stations['Passenger_Net_Flow'], 
           name='Net Passenger Flow'),
    row=2, col=2
)

fig_stations.update_layout(height=800, title_text="Station Network Analysis")
fig_stations.show()

In [ ]:
# 3. Station Analysis
# Top stations by traffic
from_station_traffic = df.groupby('From_Station').agg({
    'TripID': 'count',
    'Passengers': 'sum'
}).sort_values('TripID', ascending=False).head(15)

to_station_traffic = df.groupby('To_Station').agg({
    'TripID': 'count',
    'Passengers': 'sum'
}).sort_values('TripID', ascending=False).head(15)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Top Departure Stations', 'Top Arrival Stations')
)

fig.add_trace(go.Bar(x=from_station_traffic['TripID'], y=from_station_traffic.index, 
                   orientation='h', name='Departures'), row=1, col=1)

fig.add_trace(go.Bar(x=to_station_traffic['TripID'], y=to_station_traffic.index, 
                   orientation='h', name='Arrivals', marker_color='orange'), row=1, col=2)

fig.update_layout(height=500, showlegend=False, title_text="Station Traffic Analysis")
fig.show()

In [ ]:
# 4. Ticket Type Analysis
ticket_analysis = df.groupby('Ticket_Type').agg({
    'TripID': 'count',
    'Passengers': 'sum',
    'Total_Revenue': 'sum',
    'Fare': 'mean'
}).reset_index()

fig = px.sunburst(ticket_analysis, path=['Ticket_Type'], values='TripID',
                 title='Ticket Type Distribution by Number of Trips',
                 hover_data=['Passengers', 'Total_Revenue'])
fig.show()

# Ticket type performance metrics
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Trips by Ticket Type', 'Passengers by Ticket Type',
                   'Revenue by Ticket Type', 'Average Fare by Ticket Type')
)

fig.add_trace(go.Bar(x=ticket_analysis['Ticket_Type'], y=ticket_analysis['TripID'], 
                   name='Trips'), row=1, col=1)

fig.add_trace(go.Bar(x=ticket_analysis['Ticket_Type'], y=ticket_analysis['Passengers'], 
                   name='Passengers'), row=1, col=2)

fig.add_trace(go.Bar(x=ticket_analysis['Ticket_Type'], y=ticket_analysis['Total_Revenue'], 
                   name='Revenue'), row=2, col=1)

fig.add_trace(go.Bar(x=ticket_analysis['Ticket_Type'], y=ticket_analysis['Fare'], 
                   name='Avg Fare'), row=2, col=2)

fig.update_layout(height=600, showlegend=False, title_text="Ticket Type Analysis")
fig.show()

In [ ]:
# 5. Remarks/Event Analysis
remarks_analysis = df.groupby('Remarks').agg({
    'TripID': 'count',
    'Passengers': 'sum',
    'Total_Revenue': 'sum',
    'Fare': 'mean'
}).reset_index()

fig = px.pie(remarks_analysis, values='TripID', names='Remarks', 
             title='Distribution of Trips by Remarks/Events')
fig.show()

# Impact of different remarks on metrics
fig = go.Figure()

fig.add_trace(go.Bar(name='Number of Trips', x=remarks_analysis['Remarks'], y=remarks_analysis['TripID']))
fig.add_trace(go.Bar(name='Total Passengers', x=remarks_analysis['Remarks'], y=remarks_analysis['Passengers']))
fig.add_trace(go.Bar(name='Total Revenue', x=remarks_analysis['Remarks'], y=remarks_analysis['Total_Revenue']))

fig.update_layout(barmode='group', title='Impact of Remarks/Events on Metro Usage',
                 xaxis_title='Remarks/Events', yaxis_title='Count/Amount')
fig.show()

In [ ]:
# 6. Correlation Analysis
# Select numerical columns for correlation
numerical_cols = ['Distance_km', 'Fare', 'Cost_per_passenger', 'Passengers', 'Total_Revenue']
correlation_matrix = df[numerical_cols].corr()

fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale='RdBu',
    zmin=-1, zmax=1,
    hoverongaps=False,
    text=correlation_matrix.round(2),
    texttemplate="%{text}"
))

fig.update_layout(title='Correlation Matrix of Numerical Variables')
fig.show()

In [ ]:
# 7. Seasonal and Day-of-Week Analysis
# Day of week analysis
dow_analysis = df.groupby('DayOfWeek').agg({
    'TripID': 'count',
    'Passengers': 'sum',
    'Total_Revenue': 'sum'
}).reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])

# Monthly seasonality
monthly_seasonality = df.groupby('Month').agg({
    'TripID': 'count',
    'Passengers': 'sum',
    'Total_Revenue': 'sum'
})

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Trips by Day of Week', 'Passengers by Day of Week',
                   'Revenue by Day of Week', 'Monthly Seasonality')
)

# Day of week plots
fig.add_trace(go.Bar(x=dow_analysis.index, y=dow_analysis['TripID'], name='Trips'), row=1, col=1)
fig.add_trace(go.Bar(x=dow_analysis.index, y=dow_analysis['Passengers'], name='Passengers'), row=1, col=2)
fig.add_trace(go.Bar(x=dow_analysis.index, y=dow_analysis['Total_Revenue'], name='Revenue'), row=2, col=1)

# Monthly seasonality
fig.add_trace(go.Scatter(x=monthly_seasonality.index, y=monthly_seasonality['Passengers'], 
                       mode='lines+markers', name='Monthly Passengers'), row=2, col=2)

fig.update_layout(height=600, showlegend=False, title_text="Seasonal and Day-of-Week Analysis")
fig.show()

In [ ]:
df['Distance_km'] = pd.to_numeric(df['Distance_km'], errors='coerce')
df['Cost_per_passenger'] = pd.to_numeric(df['Cost_per_passenger'], errors='coerce')
df['Passengers'] = pd.to_numeric(df['Passengers'], errors='coerce')

clean_df = df.dropna(subset=['Distance_km', 'Cost_per_passenger', 'Passengers'])

In [ ]:
# 8. Distance vs Fare Analysis
fig = px.scatter(df, x='Distance_km', y='Fare', color='Ticket_Type',
                title='Distance vs Fare by Ticket Type',
                trendline='lowess',
                hover_data=['From_Station', 'To_Station', 'Passengers'])
fig.show()

# Cost efficiency analysis
fig = px.scatter(clean_df, x='Distance_km', y='Cost_per_passenger', color='Remarks',
                title='Distance vs Cost per Passenger by Remarks',
                size='Passengers',
                hover_data=['From_Station', 'To_Station', 'Ticket_Type'])
fig.show()

In [ ]:
# 9. Passenger Distribution Analysis
# Create passenger segments
df['Passenger_Segment'] = pd.cut(df['Passengers'], 
                               bins=[0, 10, 20, 30, float('inf')],
                               labels=['Small (1-10)', 'Medium (11-20)', 'Large (21-30)', 'Very Large (30+)'])

segment_analysis = df.groupby('Passenger_Segment').agg({
    'TripID': 'count',
    'Total_Revenue': 'sum',
    'Distance_km': 'mean',
    'Fare': 'mean'
}).reset_index()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Trips by Passenger Segment', 'Revenue by Passenger Segment',
                   'Avg Distance by Segment', 'Avg Fare by Segment')
)

fig.add_trace(go.Bar(x=segment_analysis['Passenger_Segment'], y=segment_analysis['TripID']), row=1, col=1)
fig.add_trace(go.Bar(x=segment_analysis['Passenger_Segment'], y=segment_analysis['Total_Revenue']), row=1, col=2)
fig.add_trace(go.Bar(x=segment_analysis['Passenger_Segment'], y=segment_analysis['Distance_km']), row=2, col=1)
fig.add_trace(go.Bar(x=segment_analysis['Passenger_Segment'], y=segment_analysis['Fare']), row=2, col=2)

fig.update_layout(height=600, showlegend=False, title_text="Passenger Segment Analysis")
fig.show()

In [ ]:
# 10. Comprehensive Summary Dashboard
# Create a summary statistics table
summary_stats = pd.DataFrame({
    'Metric': ['Total Trips', 'Total Passengers', 'Total Revenue', 'Average Fare', 
               'Average Distance', 'Average Passengers per Trip', 'Date Range', 'Unique Stations'],
    'Value': [len(df), df['Passengers'].sum(), df['Total_Revenue'].sum(), 
             round(df['Fare'].mean(), 2), round(df['Distance_km'].mean(), 2), 
             round(df['Passengers'].mean(), 2),
             f"{df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}",
             len(set(df['From_Station'].unique()) | set(df['To_Station'].unique()))]
})

print("=== COMPREHENSIVE SUMMARY ===")
print(summary_stats)

# Final overview visualization
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Top 10 Busiest Routes', 'Revenue by Year', 'Passenger Distribution',
                   'Ticket Type Revenue Share', 'Monthly Passenger Trends', 'Remarks Impact on Revenue'),
    specs=[[{"type": "bar"}, {"type": "bar"}, {"type": "pie"}],
          [{"type": "pie"}, {"type": "scatter"}, {"type": "bar"}]]
)

# Top 10 busiest routes
route_traffic = df.groupby(['From_Station', 'To_Station']).size().reset_index(name='Count')
route_traffic['Route'] = route_traffic['From_Station'] + ' to ' + route_traffic['To_Station']
top_routes = route_traffic.nlargest(10, 'Count')
fig.add_trace(go.Bar(x=top_routes['Count'], y=top_routes['Route'], orientation='h', 
                   name='Busiest Routes'), row=1, col=1)

# Revenue by year
yearly_revenue = df.groupby('Year')['Total_Revenue'].sum().reset_index()
fig.add_trace(go.Bar(x=yearly_revenue['Year'], y=yearly_revenue['Total_Revenue'], 
                   name='Yearly Revenue'), row=1, col=2)

# Passenger distribution - check if Passenger_Segment exists, if not create it
if 'Passenger_Segment' not in df.columns:
    df['Passenger_Segment'] = pd.cut(df['Passengers'], 
                                   bins=[0, 10, 20, 30, float('inf')],
                                   labels=['Small (1-10)', 'Medium (11-20)', 'Large (21-30)', 'Very Large (30+)'])

passenger_dist = df['Passenger_Segment'].value_counts()
fig.add_trace(go.Pie(labels=passenger_dist.index, values=passenger_dist.values, 
                   name='Passenger Segments'), row=1, col=3)

# Ticket type revenue share
ticket_revenue = df.groupby('Ticket_Type')['Total_Revenue'].sum().reset_index()
fig.add_trace(go.Pie(labels=ticket_revenue['Ticket_Type'], values=ticket_revenue['Total_Revenue'], 
                   name='Ticket Revenue'), row=2, col=1)

# Monthly passenger trends - ensure monthly_data exists
if 'monthly_data' not in locals():
    monthly_data = df.groupby('MonthYear').agg({
        'Passengers': 'sum',
        'Total_Revenue': 'sum',
        'TripID': 'count'
    }).reset_index()
    monthly_data['MonthYear'] = monthly_data['MonthYear'].astype(str)

fig.add_trace(go.Scatter(x=monthly_data['MonthYear'], y=monthly_data['Passengers'], 
                       mode='lines+markers', name='Monthly Passengers'), row=2, col=2)

# Remarks impact on revenue
remarks_revenue = df.groupby('Remarks')['Total_Revenue'].sum().reset_index()
fig.add_trace(go.Bar(x=remarks_revenue['Remarks'], y=remarks_revenue['Total_Revenue'], 
                   name='Revenue by Remarks'), row=2, col=3)

# Update layout and axes
fig.update_layout(
    height=800, 
    showlegend=False, 
    title_text="Delhi Metro Comprehensive Analysis Dashboard"
)

# Update axes labels
fig.update_xaxes(title_text="Number of Trips", row=1, col=1)
fig.update_xaxes(title_text="Year", row=1, col=2)
fig.update_xaxes(title_text="Remarks", row=2, col=3)
fig.update_xaxes(title_text="Month-Year", row=2, col=2, tickangle=45)

fig.update_yaxes(title_text="Routes", row=1, col=1)
fig.update_yaxes(title_text="Revenue", row=1, col=2)
fig.update_yaxes(title_text="Revenue", row=2, col=3)
fig.update_yaxes(title_text="Passengers", row=2, col=2)

fig.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Create daily time series
daily_data = df.groupby('Date').agg({
    'Passengers': 'sum',
    'Total_Revenue': 'sum',
    'TripID': 'count',
    'Fare': 'mean'
}).reset_index()

# Set Date as index
daily_data.set_index('Date', inplace=True)

# Create monthly time series for more stable forecasting
monthly_ts = df.groupby('MonthYear').agg({
    'Passengers': 'sum',
    'Total_Revenue': 'sum',
    'TripID': 'count'
}).reset_index()
monthly_ts['MonthYear'] = monthly_ts['MonthYear'].astype(str)
monthly_ts.set_index(pd.to_datetime(monthly_ts['MonthYear']), inplace=True)

print("Time Series Data Prepared:")
print(f"Daily data shape: {daily_data.shape}")
print(f"Monthly data shape: {monthly_ts.shape}")

In [ ]:
# 1. Time Series Decomposition and Stationarity Analysis
def check_stationarity(timeseries):
    """Check stationarity using Augmented Dickey-Fuller test"""
    result = adfuller(timeseries.dropna())
    print(f'ADF Statistic: {result[0]:.6f}')
    print(f'p-value: {result[1]:.6f}')
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'\t{key}: {value:.3f}')
    
    if result[1] <= 0.05:
        print("Series is stationary")
        return True
    else:
        print("Series is not stationary")
        return False

# Analyze passenger time series
passengers_ts = daily_data['Passengers']

print("=== STATIONARITY ANALYSIS ===")
print("Original Passenger Series:")
stationary = check_stationarity(passengers_ts)

# Plot original series and its components
fig = make_subplots(rows=4, cols=1, subplot_titles=['Original Series', 'Trend', 'Seasonal', 'Residual'])

# Since we have daily data, let's use a period of 7 for weekly seasonality
try:
    decomposition = seasonal_decompose(passengers_ts, model='additive', period=7)
    
    fig.add_trace(go.Scatter(x=passengers_ts.index, y=passengers_ts, name='Original'), row=1, col=1)
    fig.add_trace(go.Scatter(x=decomposition.trend.index, y=decomposition.trend, name='Trend'), row=2, col=1)
    fig.add_trace(go.Scatter(x=decomposition.seasonal.index, y=decomposition.seasonal, name='Seasonal'), row=3, col=1)
    fig.add_trace(go.Scatter(x=decomposition.resid.index, y=decomposition.resid, name='Residual'), row=4, col=1)
    
    fig.update_layout(height=800, title_text="Time Series Decomposition - Daily Passengers")
    fig.show()
    
except Exception as e:
    print(f"Decomposition error: {e}")
    # Use monthly data instead
    monthly_passengers = monthly_ts['Passengers']
    decomposition = seasonal_decompose(monthly_passengers, model='additive', period=12)
    
    fig = make_subplots(rows=4, cols=1, subplot_titles=['Original Series', 'Trend', 'Seasonal', 'Residual'])
    fig.add_trace(go.Scatter(x=monthly_passengers.index, y=monthly_passengers, name='Original'), row=1, col=1)
    fig.add_trace(go.Scatter(x=decomposition.trend.index, y=decomposition.trend, name='Trend'), row=2, col=1)
    fig.add_trace(go.Scatter(x=decomposition.seasonal.index, y=decomposition.seasonal, name='Seasonal'), row=3, col=1)
    fig.add_trace(go.Scatter(x=decomposition.resid.index, y=decomposition.resid, name='Residual'), row=4, col=1)
    fig.update_layout(height=800, title_text="Time Series Decomposition - Monthly Passengers")
    fig.show()

In [ ]:
# 2. ARIMA Modeling
def evaluate_arima_model(data, order, train_size=0.8):
    """Train and evaluate ARIMA model"""
    # Split data
    split_idx = int(len(data) * train_size)
    train, test = data[:split_idx], data[split_idx:]
    
    # Fit ARIMA model
    model = ARIMA(train, order=order)
    model_fit = model.fit()
    
    # Forecast
    forecast = model_fit.forecast(steps=len(test))
    
    # Calculate metrics
    mse = mean_squared_error(test, forecast)
    mae = mean_absolute_error(test, forecast)
    rmse = np.sqrt(mse)
    
    return model_fit, forecast, test, mse, mae, rmse

# Try different ARIMA orders
print("=== ARIMA MODELING ===")

# Use monthly data for more stable modeling
monthly_passengers = monthly_ts['Passengers']

# Test multiple ARIMA configurations
arima_orders = [(1,1,1), (2,1,2), (1,1,0), (0,1,1)]
best_arima = None
best_arima_rmse = float('inf')

for order in arima_orders:
    try:
        model_fit, forecast, test, mse, mae, rmse = evaluate_arima_model(monthly_passengers, order)
        print(f"ARIMA{order} - RMSE: {rmse:.2f}, MAE: {mae:.2f}")
        
        if rmse < best_arima_rmse:
            best_arima_rmse = rmse
            best_arima = (order, model_fit, forecast, test, rmse)
    except Exception as e:
        print(f"ARIMA{order} failed: {e}")

if best_arima:
    order, model_fit, forecast, test, rmse = best_arima
    print(f"\nBest ARIMA model: ARIMA{order} with RMSE: {rmse:.2f}")
    
    # Plot ARIMA results
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=monthly_passengers.index, y=monthly_passengers, 
                           name='Actual', line=dict(color='blue')))
    fig.add_trace(go.Scatter(x=test.index, y=forecast, 
                           name='ARIMA Forecast', line=dict(color='red', dash='dash')))
    fig.update_layout(title=f'ARIMA{order} Forecasting - Monthly Passengers',
                    xaxis_title='Date', yaxis_title='Passengers')
    fig.show()
    
    # Print model summary
    print("\nARIMA Model Summary:")
    print(model_fit.summary())

In [ ]:
# 3. SARIMA Modeling
def evaluate_sarima_model(data, order, seasonal_order, train_size=0.8):
    """Train and evaluate SARIMA model"""
    # Split data
    split_idx = int(len(data) * train_size)
    train, test = data[:split_idx], data[split_idx:]
    
    # Fit SARIMA model
    model = SARIMAX(train, order=order, seasonal_order=seasonal_order)
    model_fit = model.fit(disp=False)
    
    # Forecast
    forecast = model_fit.forecast(steps=len(test))
    
    # Calculate metrics
    mse = mean_squared_error(test, forecast)
    mae = mean_absolute_error(test, forecast)
    rmse = np.sqrt(mse)
    
    return model_fit, forecast, test, mse, mae, rmse

print("\n=== SARIMA MODELING ===")

# Test SARIMA configurations (using monthly data with yearly seasonality)
sarima_configs = [
    ((1,1,1), (1,1,1,12)),  # ARIMA(1,1,1) with seasonal (1,1,1,12)
    ((1,1,0), (1,1,0,12)),  # ARIMA(1,1,0) with seasonal (1,1,0,12)
    ((0,1,1), (0,1,1,12)),  # ARIMA(0,1,1) with seasonal (0,1,1,12)
    ((2,1,2), (1,1,1,12)),  # ARIMA(2,1,2) with seasonal (1,1,1,12)
]

best_sarima = None
best_sarima_rmse = float('inf')

for order, seasonal_order in sarima_configs:
    try:
        model_fit, forecast, test, mse, mae, rmse = evaluate_sarima_model(
            monthly_passengers, order, seasonal_order)
        print(f"SARIMA{order}x{seasonal_order} - RMSE: {rmse:.2f}, MAE: {mae:.2f}")
        
        if rmse < best_sarima_rmse:
            best_sarima_rmse = rmse
            best_sarima = (order, seasonal_order, model_fit, forecast, test, rmse)
    except Exception as e:
        print(f"SARIMA{order}x{seasonal_order} failed: {e}")

if best_sarima:
    order, seasonal_order, model_fit, forecast, test, rmse = best_sarima
    print(f"\nBest SARIMA model: SARIMA{order}x{seasonal_order} with RMSE: {rmse:.2f}")
    
    # Plot SARIMA results
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=monthly_passengers.index, y=monthly_passengers, 
                           name='Actual', line=dict(color='blue')))
    fig.add_trace(go.Scatter(x=test.index, y=forecast, 
                           name='SARIMA Forecast', line=dict(color='green', dash='dash')))
    fig.update_layout(title=f'SARIMA{order}x{seasonal_order} Forecasting - Monthly Passengers',
                    xaxis_title='Date', yaxis_title='Passengers')
    fig.show()
    
    # Print model summary
    print("\nSARIMA Model Summary:")
    print(model_fit.summary())

# FORECASTING

In [ ]:
# 4. Future Forecasting
def forecast_future(model_fit, steps, is_sarima=True):
    """Generate future forecasts"""
    if is_sarima:
        forecast = model_fit.forecast(steps=steps)
        conf_int = model_fit.get_forecast(steps=steps).conf_int()
    else:
        forecast = model_fit.forecast(steps=steps)
        # For ARIMA, we need to calculate confidence intervals manually
        conf_int = None
    
    return forecast, conf_int

# Use the best model for future forecasting
if best_sarima:
    best_model = best_sarima[2]  # SARIMA model
    model_type = "SARIMA"
    order_info = f"{best_sarima[0]}x{best_sarima[1]}"
elif best_arima:
    best_model = best_arima[1]  # ARIMA model
    model_type = "ARIMA"
    order_info = f"{best_arima[0]}"
else:
    # Fallback to simple ARIMA
    best_model = ARIMA(monthly_passengers, order=(1,1,1)).fit()
    model_type = "ARIMA"
    order_info = "(1,1,1)"

# Forecast next 24 months
future_steps = 24
future_forecast, conf_int = forecast_future(best_model, future_steps, is_sarima=(model_type=="SARIMA"))

# Create future dates
last_date = monthly_passengers.index[-1]
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), 
                           periods=future_steps, freq='M')

# Plot future forecasts
fig = go.Figure()

# Historical data
fig.add_trace(go.Scatter(x=monthly_passengers.index, y=monthly_passengers,
                        name='Historical', line=dict(color='blue')))

# Future forecast
fig.add_trace(go.Scatter(x=future_dates, y=future_forecast,
                        name=f'{model_type} Forecast', line=dict(color='red')))

# Confidence intervals if available
if conf_int is not None:
    fig.add_trace(go.Scatter(x=future_dates, y=conf_int.iloc[:, 0],
                           name='Lower CI', line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=future_dates, y=conf_int.iloc[:, 1],
                           name='Upper CI', line=dict(width=0), 
                           fillcolor='rgba(255,0,0,0.2)', fill='tonexty',
                           showlegend=False))

fig.update_layout(title=f'Future Passenger Forecast - {model_type}{order_info}',
                xaxis_title='Date', yaxis_title='Passengers',
                height=500)
fig.show()

print(f"\n=== FUTURE FORECAST - Next {future_steps} Months ===")
forecast_df = pd.DataFrame({
    'Month': future_dates.strftime('%Y-%m'),
    'Forecasted_Passengers': future_forecast.round(0)
})
print(forecast_df)

In [ ]:
# 5. Model Comparison and Residual Analysis
def plot_residuals(model_fit, model_name):
    """Plot residual analysis"""
    residuals = model_fit.resid.dropna()

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'Residuals over Time', 
            'Residual Distribution',
            'Q-Q Plot', 
            'ACF of Residuals'
        ]
    )

    # Residuals over time
    fig.add_trace(
        go.Scatter(
            x=list(range(len(residuals))),  # FIXED: convert range → list
            y=residuals,
            mode='lines',
            name='Residuals'
        ),
        row=1, col=1
    )

    # Residual distribution
    fig.add_trace(
        go.Histogram(x=residuals, nbinsx=30, name='Distribution'),
        row=1, col=2
    )

    # Q-Q plot
    from scipy import stats
    qq = stats.probplot(residuals, dist="norm")
    fig.add_trace(
        go.Scatter(
            x=qq[0][0],
            y=qq[0][1],
            mode='markers',
            name='Q-Q Plot'
        ),
        row=2, col=1
    )

    # ACF of residuals
    from statsmodels.tsa.stattools import acf
    acf_vals = acf(residuals, nlags=20)
    fig.add_trace(
        go.Bar(
            x=list(range(len(acf_vals))),
            y=acf_vals,
            name='ACF'
        ),
        row=2, col=2
    )

    fig.update_layout(height=600, title_text=f'Residual Analysis - {model_name}')
    fig.show()

    # Ljung-Box test
    from statsmodels.stats.diagnostic import acorr_ljungbox
    lb_test = acorr_ljungbox(residuals, lags=10)
    print(f"\nLjung-Box Test p-value: {lb_test['lb_pvalue'].iloc[-1]:.4f}")


# Analyze residuals for best model
if best_sarima:
    plot_residuals(best_sarima[2], f"SARIMA{best_sarima[0]}x{best_sarima[1]}")
elif best_arima:
    plot_residuals(best_arima[1], f"ARIMA{best_arima[0]}")

In [ ]:
# 6. Multi-variable Time Series Analysis
print("\n=== MULTI-VARIABLE TIME SERIES ANALYSIS ===")

# Prepare multiple time series
multi_ts = monthly_ts[['Passengers', 'Total_Revenue', 'TripID']]

# Correlation analysis
correlation_matrix = multi_ts.corr()
fig = px.imshow(correlation_matrix, text_auto=True, aspect="auto",
               title="Correlation Matrix - Monthly Metrics")
fig.show()

# Multi-series forecasting using VAR (Vector Auto Regression)
from statsmodels.tsa.vector_ar.var_model import VAR

# Prepare data for VAR
var_data = multi_ts.dropna()

# Split for VAR modeling
train_size = int(len(var_data) * 0.8)
var_train = var_data[:train_size]
var_test = var_data[train_size:]

try:
    # Fit VAR model
    var_model = VAR(var_train)
    var_result = var_model.fit(maxlags=4, ic='aic')
    
    print("VAR Model Summary:")
    print(var_result.summary())
    
    # Forecast
    lag_order = var_result.k_ar
    var_forecast = var_result.forecast(var_train.values[-lag_order:], steps=len(var_test))
    
    # Create forecast dataframe
    forecast_df = pd.DataFrame(var_forecast, index=var_test.index, columns=var_test.columns)
    
    # Plot multi-series forecast
    fig = make_subplots(rows=3, cols=1, subplot_titles=['Passengers', 'Revenue', 'Trips'])
    
    for i, col in enumerate(['Passengers', 'Total_Revenue', 'TripID']):
        fig.add_trace(go.Scatter(x=var_data.index, y=var_data[col], 
                               name=f'Actual {col}', line=dict(color='blue')), row=i+1, col=1)
        fig.add_trace(go.Scatter(x=forecast_df.index, y=forecast_df[col], 
                               name=f'Forecast {col}', line=dict(color='red', dash='dash')), row=i+1, col=1)
    
    fig.update_layout(height=800, title_text="Multi-Variable Time Series Forecast (VAR)")
    fig.show()
    
except Exception as e:
    print(f"VAR modeling failed: {e}")

In [ ]:
# 7. Performance Metrics and Model Selection
print("\n=== MODEL PERFORMANCE COMPARISON ===")

models_performance = []

# Collect ARIMA performance
if best_arima:
    models_performance.append({
        'Model': f'ARIMA{best_arima[0]}',
        'RMSE': best_arima_rmse,
        'Type': 'ARIMA'
    })

# Collect SARIMA performance
if best_sarima:
    models_performance.append({
        'Model': f'SARIMA{best_sarima[0]}x{best_sarima[1]}',
        'RMSE': best_sarima_rmse,
        'Type': 'SARIMA'
    })

# Create performance comparison
if models_performance:
    performance_df = pd.DataFrame(models_performance)
    fig = px.bar(performance_df, x='Model', y='RMSE', color='Type',
                title='Model Performance Comparison (Lower RMSE is Better)',
                text_auto=True)
    fig.show()
    
    # Best model recommendation
    best_model_info = performance_df.loc[performance_df['RMSE'].idxmin()]
    print(f"\nRECOMMENDED MODEL: {best_model_info['Model']}")
    print(f"RMSE: {best_model_info['RMSE']:.2f}")
    
    # Business insights from forecasts
    print("\n=== BUSINESS INSIGHTS ===")
    avg_monthly_growth = (future_forecast[-1] - future_forecast[0]) / future_forecast[0] * 100 / (future_steps - 1)
    print(f"Average monthly growth rate: {avg_monthly_growth:.2f}%")
    print(f"Expected passengers in {future_dates[-1].strftime('%Y-%m')}: {future_forecast[-1]:.0f}")

In [ ]:
# POLICY EVALUATION FRAMEWORK
print("=== DELHI METRO POLICY EVALUATION FRAMEWORK ===")

# 1. Policy Impact Analysis
def evaluate_policy_impact(df, policy_metric, target_metric):
    """Evaluate impact of specific policies on target metrics"""
    policy_analysis = df.groupby(policy_metric).agg({
        target_metric: ['mean', 'sum', 'count'],
        'Total_Revenue': 'mean',
        'Passengers': 'mean',
        'Distance_km': 'mean'
    }).round(2)
    
    return policy_analysis

# Policy 1: Ticket Type Analysis
print("\n1. TICKET TYPE POLICY EVALUATION")
ticket_policy = evaluate_policy_impact(df, 'Ticket_Type', 'Fare')
print(ticket_policy)

# Policy 2: Time-based Pricing (Remarks Analysis)
print("\n2. TIME-BASED PRICING POLICY EVALUATION")
time_policy = evaluate_policy_impact(df, 'Remarks', 'Fare')
print(time_policy)

# Policy 3: Station Performance Analysis
print("\n3. STATION PERFORMANCE POLICY EVALUATION")
station_revenue = df.groupby('From_Station').agg({
    'Total_Revenue': 'sum',
    'Passengers': 'sum',
    'TripID': 'count',
    'Fare': 'mean'
}).sort_values('Total_Revenue', ascending=False).head(10)
print("Top 10 Stations by Revenue:")
print(station_revenue)

In [ ]:
# 2. Comprehensive Policy Analysis Dashboard
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Revenue by Ticket Type', 'Passenger Distribution by Ticket Type',
        'Fare Structure Analysis', 'Time-based Pricing Impact',
        'Station Revenue Performance', 'Cost Efficiency by Policy'
    ),
    specs=[[{"type": "bar"}, {"type": "pie"}, {"type": "bar"}],
          [{"type": "bar"}, {"type": "bar"}, {"type": "scatter"}]]
)

# Plot 1: Revenue by Ticket Type
ticket_revenue = df.groupby('Ticket_Type')['Total_Revenue'].sum()
fig.add_trace(go.Bar(x=ticket_revenue.index, y=ticket_revenue.values, name='Revenue'), row=1, col=1)

# Plot 2: Passenger Distribution by Ticket Type
ticket_passengers = df.groupby('Ticket_Type')['Passengers'].sum()
fig.add_trace(go.Pie(labels=ticket_passengers.index, values=ticket_passengers.values, name='Passengers'), row=1, col=2)

# Plot 3: Fare Structure Analysis
avg_fare_by_ticket = df.groupby('Ticket_Type')['Fare'].mean()
fig.add_trace(go.Bar(x=avg_fare_by_ticket.index, y=avg_fare_by_ticket.values, name='Avg Fare'), row=1, col=3)

# Plot 4: Time-based Pricing Impact
revenue_by_remarks = df.groupby('Remarks')['Total_Revenue'].sum()
fig.add_trace(go.Bar(x=revenue_by_remarks.index, y=revenue_by_remarks.values, name='Revenue by Time'), row=2, col=1)

# Plot 5: Station Revenue Performance
fig.add_trace(go.Bar(x=station_revenue.index, y=station_revenue['Total_Revenue'], name='Station Revenue'), row=2, col=2)

# Plot 6: Cost Efficiency by Policy
cost_efficiency = df.groupby('Ticket_Type').agg({
    'Cost_per_passenger': 'mean',
    'Total_Revenue': 'sum'
}).reset_index()
fig.add_trace(go.Scatter(x=cost_efficiency['Cost_per_passenger'], 
                        y=cost_efficiency['Total_Revenue'],
                        mode='markers+text',
                        text=cost_efficiency['Ticket_Type'],
                        marker=dict(size=cost_efficiency['Total_Revenue']/10000),
                        name='Cost Efficiency'), row=2, col=3)

fig.update_layout(height=800, title_text="Comprehensive Policy Evaluation Dashboard", showlegend=False)
fig.show()

In [ ]:
# 3. Revenue Optimization Analysis
print("\n=== REVENUE OPTIMIZATION ANALYSIS ===")

# Calculate current revenue metrics
total_revenue = df['Total_Revenue'].sum()
avg_fare = df['Fare'].mean()
total_passengers = df['Passengers'].sum()

print(f"Current Total Revenue: ₹{total_revenue:,.2f}")
print(f"Current Average Fare: ₹{avg_fare:.2f}")
print(f"Total Passengers: {total_passengers:,}")

# Scenario Analysis: Fare Adjustment Impact
def simulate_fare_adjustment(df, fare_increase_pct, demand_elasticity=-0.3):
    """
    Simulate impact of fare adjustments
    demand_elasticity: % change in demand for 1% change in fare (typically negative)
    """
    new_fare = df['Fare'] * (1 + fare_increase_pct/100)
    demand_change = fare_increase_pct * demand_elasticity
    new_passengers = df['Passengers'] * (1 + demand_change/100)
    new_revenue = new_fare * new_passengers
    
    current_revenue = (df['Fare'] * df['Passengers']).sum()
    projected_revenue = new_revenue.sum()
    revenue_change_pct = (projected_revenue - current_revenue) / current_revenue * 100
    
    return {
        'fare_increase_pct': fare_increase_pct,
        'current_revenue': current_revenue,
        'projected_revenue': projected_revenue,
        'revenue_change_pct': revenue_change_pct,
        'passenger_change_pct': demand_change,
        'new_avg_fare': new_fare.mean()
    }

# Test different fare adjustment scenarios
fare_scenarios = []
for increase_pct in [0, 5, 10, 15, 20, -5, -10]:
    scenario = simulate_fare_adjustment(df, increase_pct)
    fare_scenarios.append(scenario)

fare_scenarios_df = pd.DataFrame(fare_scenarios)
print("\nFare Adjustment Scenario Analysis:")
print(fare_scenarios_df.round(2))

# Plot fare adjustment impact
fig = go.Figure()
fig.add_trace(go.Scatter(x=fare_scenarios_df['fare_increase_pct'], 
                        y=fare_scenarios_df['revenue_change_pct'],
                        mode='lines+markers',
                        name='Revenue Impact',
                        line=dict(color='green')))
fig.add_trace(go.Scatter(x=fare_scenarios_df['fare_increase_pct'], 
                        y=fare_scenarios_df['passenger_change_pct'],
                        mode='lines+markers',
                        name='Passenger Impact',
                        line=dict(color='red')))

fig.update_layout(title='Fare Adjustment Impact Analysis',
                 xaxis_title='Fare Change Percentage (%)',
                 yaxis_title='Impact Percentage (%)',
                 hovermode='x unified')
fig.show()

In [ ]:
# 4. Operational Efficiency Analysis
print("\n=== OPERATIONAL EFFICIENCY ANALYSIS ===")

# Route Efficiency Analysis
route_efficiency = df.groupby(['From_Station', 'To_Station']).agg({
    'TripID': 'count',
    'Passengers': 'sum',
    'Total_Revenue': 'sum',
    'Distance_km': 'mean',
    'Cost_per_passenger': 'mean'
}).reset_index()

route_efficiency['Revenue_per_km'] = route_efficiency['Total_Revenue'] / route_efficiency['Distance_km']
route_efficiency['Passengers_per_trip'] = route_efficiency['Passengers'] / route_efficiency['TripID']

# Identify high-performing and underperforming routes
high_performing = route_efficiency.nlargest(10, 'Revenue_per_km')
underperforming = route_efficiency[route_efficiency['TripID'] > 5].nsmallest(10, 'Revenue_per_km')

print("Top 10 High-Performing Routes:")
print(high_performing[['From_Station', 'To_Station', 'Revenue_per_km', 'Passengers_per_trip']].round(2))

print("\nTop 10 Underperforming Routes:")
print(underperforming[['From_Station', 'To_Station', 'Revenue_per_km', 'Passengers_per_trip']].round(2))

# Capacity Utilization Analysis
capacity_analysis = df.groupby('Passenger_Segment').agg({
    'TripID': 'count',
    'Total_Revenue': 'sum',
    'Fare': 'mean'
}).reset_index()

print("\nCapacity Utilization Analysis:")
print(capacity_analysis)

In [ ]:
# 5. Time-based Policy Evaluation
print("\n=== TIME-BASED POLICY EVALUATION ===")

# Analyze different time periods (Remarks)
time_period_analysis = df.groupby('Remarks').agg({
    'TripID': 'count',
    'Passengers': 'sum',
    'Total_Revenue': 'sum',
    'Fare': 'mean',
    'Cost_per_passenger': 'mean'
}).round(2)

print("Time Period Performance Analysis:")
print(time_period_analysis)

# Peak vs Off-Peak Analysis
peak_data = df[df['Remarks'].isin(['peak', 'off-peak'])]
peak_analysis = peak_data.groupby('Remarks').agg({
    'Passengers': 'sum',
    'Total_Revenue': 'sum',
    'Fare': 'mean',
    'TripID': 'count'
})

print("\nPeak vs Off-Peak Analysis:")
print(peak_analysis)

# Calculate peak hour efficiency
peak_efficiency = peak_analysis['Total_Revenue'] / peak_analysis['TripID']
print(f"\nRevenue per Trip - Peak: ₹{peak_efficiency['peak']:.2f}, Off-Peak: ₹{peak_efficiency['off-peak']:.2f}")

# Dynamic Pricing Opportunity Analysis
current_peak_premium = (peak_analysis.loc['peak', 'Fare'] - peak_analysis.loc['off-peak', 'Fare']) / peak_analysis.loc['off-peak', 'Fare'] * 100
print(f"Current Peak Premium: {current_peak_premium:.1f}%")

# Recommend optimal peak pricing
optimal_peak_premium = 25  # Industry standard
recommended_peak_fare = peak_analysis.loc['off-peak', 'Fare'] * (1 + optimal_peak_premium/100)
print(f"Recommended Peak Fare: ₹{recommended_peak_fare:.2f} (Current: ₹{peak_analysis.loc['peak', 'Fare']:.2f})")

In [ ]:
# 6. Customer Segmentation and Targeted Policies
print("\n=== CUSTOMER SEGMENTATION ANALYSIS ===")

# Create customer segments based on travel patterns
customer_segments = df.groupby(['Ticket_Type', 'Passenger_Segment']).agg({
    'TripID': 'count',
    'Total_Revenue': 'sum',
    'Fare': 'mean',
    'Distance_km': 'mean'
}).reset_index()

print("Customer Segment Analysis:")
print(customer_segments)

# Calculate customer lifetime value (simplified)
segment_clv = customer_segments.groupby('Ticket_Type').agg({
    'Total_Revenue': 'sum',
    'TripID': 'sum'
})
segment_clv['Avg_Revenue_per_Trip'] = segment_clv['Total_Revenue'] / segment_clv['TripID']
segment_clv['Customer_Value_Score'] = segment_clv['Avg_Revenue_per_Trip'] * segment_clv['TripID'] / segment_clv['TripID'].sum()

print("\nCustomer Lifetime Value Analysis:")
print(segment_clv.round(2))

# Targeted Policy Recommendations
print("\n=== TARGETED POLICY RECOMMENDATIONS ===")

# 1. Tourist Card Optimization
tourist_data = df[df['Ticket_Type'] == 'Tourist Card']
tourist_revenue_per_passenger = tourist_data['Total_Revenue'].sum() / tourist_data['Passengers'].sum()
print(f"Tourist Card Revenue per Passenger: ₹{tourist_revenue_per_passenger:.2f}")

# 2. Smart Card Loyalty Program
smart_card_users = df[df['Ticket_Type'] == 'Smart Card']
avg_smart_card_fare = smart_card_users['Fare'].mean()
print(f"Smart Card Average Fare: ₹{avg_smart_card_fare:.2f}")

# 3. Single Journey Optimization
single_journey = df[df['Ticket_Type'] == 'Single']
single_journey_revenue = single_journey['Total_Revenue'].sum()
print(f"Single Journey Total Revenue: ₹{single_journey_revenue:,.2f}")


In [ ]:
# 7. Cost-Benefit Analysis of Policy Changes
print("\n=== COST-BENEFIT ANALYSIS ===")

def calculate_policy_roi(current_metrics, proposed_metrics, implementation_cost):
    """Calculate ROI for policy changes"""
    revenue_impact = proposed_metrics['revenue'] - current_metrics['revenue']
    passenger_impact = proposed_metrics['passengers'] - current_metrics['passengers']
    
    roi = (revenue_impact - implementation_cost) / implementation_cost * 100
    payback_period = implementation_cost / revenue_impact if revenue_impact > 0 else float('inf')
    
    return {
        'revenue_impact': revenue_impact,
        'passenger_impact': passenger_impact,
        'roi_percentage': roi,
        'payback_period_months': payback_period * 12,  # Convert to months
        'net_benefit': revenue_impact - implementation_cost
    }

# Current baseline metrics
current_baseline = {
    'revenue': df['Total_Revenue'].sum(),
    'passengers': df['Passengers'].sum()
}

# Policy 1: Smart Card Loyalty Program
smart_card_loyalty = {
    'revenue': current_baseline['revenue'] * 1.15,  # 15% increase from loyalty
    'passengers': current_baseline['passengers'] * 1.05  # 5% passenger growth
}
loyalty_roi = calculate_policy_roi(current_baseline, smart_card_loyalty, 5000000)  # 50 lakh implementation

# Policy 2: Dynamic Peak Pricing
dynamic_pricing = {
    'revenue': current_baseline['revenue'] * 1.08,  # 8% revenue optimization
    'passengers': current_baseline['passengers'] * 0.98  # 2% passenger loss
}
pricing_roi = calculate_policy_roi(current_baseline, dynamic_pricing, 2000000)  # 20 lakh implementation

# Policy 3: Tourist Package Enhancement
tourist_enhancement = {
    'revenue': current_baseline['revenue'] * 1.12,  # 12% revenue growth
    'passengers': current_baseline['passengers'] * 1.08  # 8% passenger growth
}
tourist_roi = calculate_policy_roi(current_baseline, tourist_enhancement, 3000000)  # 30 lakh implementation

# Create policy comparison
policies_comparison = pd.DataFrame({
    'Smart Card Loyalty': loyalty_roi,
    'Dynamic Pricing': pricing_roi,
    'Tourist Enhancement': tourist_roi
}).T

print("Policy Cost-Benefit Analysis:")
print(policies_comparison.round(2))


In [ ]:
# 8. Final Policy Recommendations and Implementation Roadmap
print("\n" + "="*60)
print("FINAL POLICY RECOMMENDATIONS")
print("="*60)

# Summary of Key Findings
print("\nKEY FINDINGS:")
print("1. Ticket Type Distribution:")
print(f"   - Tourist Cards: {ticket_passengers.get('Tourist Card', 0):,} passengers")
print(f"   - Smart Cards: {ticket_passengers.get('Smart Card', 0):,} passengers")
print(f"   - Single Journey: {ticket_passengers.get('Single', 0):,} passengers")

print("\n2. Revenue Performance:")
print(f"   - Total Revenue: ₹{total_revenue:,.2f}")
print(f"   - Average Fare: ₹{avg_fare:.2f}")
print(f"   - Revenue per Passenger: ₹{total_revenue/total_passengers:.2f}")

print("\n3. Operational Efficiency:")
print(f"   - High-performing routes: {len(high_performing)} identified")
print(f"   - Underperforming routes: {len(underperforming)} identified")

# Policy Recommendations
print("\n" + "="*60)
print("RECOMMENDED POLICIES")
print("="*60)

print("\n🚀 IMMEDIATE ACTIONS (0-3 months):")
print("1. Implement Dynamic Peak Pricing")
print(f"   - Increase peak fares to ₹{recommended_peak_fare:.2f} (+25% premium)")
print(f"   - Expected revenue impact: +8% (₹{current_baseline['revenue'] * 0.08:,.2f})")
print(f"   - ROI: {pricing_roi['roi_percentage']:.1f}%")

print("\n2. Optimize Tourist Card Pricing")
print("   - Introduce tiered tourist packages")
print("   - Add premium features (guided tours, express lanes)")

print("\n📈 MEDIUM-TERM INITIATIVES (3-12 months):")
print("1. Launch Smart Card Loyalty Program")
print("   - Points system for frequent travelers")
print("   - Partner discounts and benefits")
print(f"   - Expected ROI: {loyalty_roi['roi_percentage']:.1f}%")

print("\n2. Route Optimization")
print("   - Increase frequency on high-performing routes")
print("   - Review and optimize underperforming routes")

print("\n🎯 LONG-TERM STRATEGIES (12+ months):")
print("1. Digital Transformation")
print("   - Mobile ticketing integration")
print("   - Real-time capacity monitoring")
print("   - AI-powered demand forecasting")

print("\n2. Station Infrastructure Upgrade")
print("   - Focus on high-revenue stations")
print("   - Enhanced passenger amenities")

# Implementation Metrics
print("\n" + "="*60)
print("SUCCESS METRICS & MONITORING")
print("="*60)

print("Key Performance Indicators (KPIs):")
print("✓ Revenue per passenger")
print("✓ Passenger satisfaction scores")
print("✓ Route efficiency metrics")
print("✓ Ticket type distribution")
print("✓ Peak vs off-peak utilization")

print(f"\nTarget: Increase overall revenue by 15% within 12 months")
print(f"Current: ₹{total_revenue:,.2f}")
print(f"Target: ₹{total_revenue * 1.15:,.2f}")

In [ ]:
# 9. Risk Assessment and Mitigation
print("\n" + "="*60)
print("RISK ASSESSMENT & MITIGATION STRATEGIES")
print("="*60)

risks_mitigation = {
    'Passenger Backlash': {
        'Risk Level': 'Medium',
        'Impact': 'Decreased ridership, negative publicity',
        'Mitigation': 'Gradual implementation, clear communication, value-added services'
    },
    'Operational Complexity': {
        'Risk Level': 'Low',
        'Impact': 'System disruptions, increased costs',
        'Mitigation': 'Phased rollout, staff training, robust testing'
    },
    'Competitive Response': {
        'Risk Level': 'Low',
        'Impact': 'Market share loss',
        'Mitigation': 'Differentiated service quality, loyalty programs'
    },
    'Regulatory Challenges': {
        'Risk Level': 'Medium',
        'Impact': 'Implementation delays, compliance costs',
        'Mitigation': 'Early stakeholder engagement, regulatory compliance review'
    }
}

risk_df = pd.DataFrame(risks_mitigation).T
print(risk_df)

# Create risk matrix visualization
fig = go.Figure()

risk_levels = {'Low': 1, 'Medium': 2, 'High': 3}
colors = {'Low': 'green', 'Medium': 'orange', 'High': 'red'}

for risk, info in risks_mitigation.items():
    fig.add_trace(go.Scatter(
        x=[risk_levels[info['Risk Level']]],
        y=[list(risks_mitigation.keys()).index(risk)],
        mode='markers+text',
        marker=dict(size=20, color=colors[info['Risk Level']]),
        text=[risk],
        textposition="middle right",
        name=info['Risk Level']
    ))

fig.update_layout(
    title='Policy Implementation Risk Matrix',
    xaxis=dict(title='Risk Level', tickvals=[1, 2, 3], ticktext=['Low', 'Medium', 'High']),
    yaxis=dict(title='', showticklabels=False),
    showlegend=False,
    height=400
)
fig.show()